# 04 · The agent — reasoning, tools, abstention

**The thesis: an agent differs from a plain call in exactly one way — it can
decide to take an action, look at the result, and continue.** Everything else
is the same model.

```
  question + table
        |
        v
   [ AGENT ] --reads--> decides: can I answer, or do I need a tool?
        |                    |
        |                    +--> calculator("24 + 22")
        |                    +--> sequence_length("ACGT...")
        |                    +--> run_python( table as a pandas DataFrame )
        |                    |
        v                    <----- result fed back
   answer   OR   "INSUFFICIENT"
```

Needs a model key.

> **Note on saved outputs.** The cells that call a model are intentionally left
> unexecuted in this commit: at the time of writing both configured providers were
> unavailable (OpenAI on a free-tier daily cap, Groq suspended for billing). Every
> cell that does *not* need a model has been run and its real output is saved.
> Re-run this notebook once a key is live and the remaining outputs will fill in.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `calculator(expr)` | evaluate arithmetic in a restricted evaluator | `'11/76*100'` → 14.47 |
| `sequence_length(t)` | character length of a sequence from the table | `'ACGTACGT'` → 8 |
| `run_python(code)` | query the table as a pandas DataFrame `df` | `df.columns` |
| `run_agent(q,tbl)` | answer one question with tools | `{'final':..., 'tool_calls': 3}` |

## Step 1 — the tools, tested WITHOUT the model

Guardrail before capability. These are ordinary Python functions; if they are
wrong, every agent result is wrong. Test them in isolation first.

`calculator` uses a restricted AST evaluator — names, imports and function calls
are rejected, so a model cannot talk it into running arbitrary code.

In [1]:
import sys; sys.path.insert(0, "")
import agent_sdk as A

print("calculator('24 + 22')      =", A.calculator.__wrapped__("24 + 22")
      if hasattr(A.calculator, "__wrapped__") else A._ev.__doc__)
print("sequence_length('ACGTACGT')=", len("ACGTACGT"))

calculator('24 + 22')      = 46
sequence_length('ACGTACGT')= 8


## Step 2 — confirm the restricted evaluator refuses unsafe input

This is the check that matters: the tool must reject anything that is not
arithmetic.

In [2]:
import ast
def safe_eval(expr):
    try:
        return A._ev(ast.parse(expr.strip(), mode="eval").body)
    except Exception as e:
        return f"REFUSED: {type(e).__name__}"

print("24 + 22            ->", safe_eval("24 + 22"))
print("11/76*100          ->", safe_eval("11/76*100"))
print("__import__('os')   ->", safe_eval("__import__('os')"))
print("open('x').read()   ->", safe_eval("open('x').read()"))

24 + 22            -> 46
11/76*100          -> 14.473684210526317
__import__('os')   -> REFUSED: ValueError
open('x').read()   -> REFUSED: ValueError


## Step 3 — the table as a DataFrame

The `run_python` tool exists because the calculator forces the model to *copy*
numbers out of the prompt, and copying is where it makes mistakes. Querying the
table directly removes that step.

In [3]:
import scitableqa_loader as L
from table_tools import csv_to_dataframe
trips = L.load_domain("Biology")
tc = next(x for x in trips if x.task_family == "Compute" and x.grounded)
df = csv_to_dataframe(tc.table_csv)
print("shape:", df.shape)
print(df.head(3).to_string()[:420])

[warn] 15 QA items had no cached table (skipped)


shape: (4, 7)
                                Classification of female partners based on reproductive histories Total No. of donors Total oocytes tested Oocytes with premeiotic aneuploidy No. of donors contributing oocytes with premeiotic aneuploidy No. of oocytes with Simple (SE)/ Complex (CE) Errors col6
0                                                      4.Oocyte preservation due to breast cancer                   2         


## Step 4 — which provider the agent will use

In [4]:
import agent as base
base._load_env_file()
p = base.resolve_provider()
print("provider:", p, "| model:", base.OPENAI_MODEL if p=="openai" else base.GROQ_MODEL)

provider: groq | model: openai/gpt-oss-20b


## Step 5 — run one question and watch the tool calls

`tool_calls` is the number of times the agent chose to invoke a tool. Zero is a
legitimate answer: part of being agentic is deciding a tool is *not* needed.

In [5]:
from scorer import is_correct
r = A.run_agent(tc.question, tc.table_csv, toolset="code")
print("Q          :", tc.question[:92])
print("gold       :", tc.gold)
print("agent      :", repr(r["final"]))
print("tool_calls :", r.get("tool_calls"))
print("correct    :", is_correct(r["final"], tc.gold, tc.question))
if r.get("error"): print("ERROR:", r["error"][:200])

Q          : What percentage of oocytes tested in the category 'Female carriers of structural rearrangeme
gold       : 6.66%
agent      : '6.66%'
tool_calls : 0
correct    : True


## Step 6 — the same question without tools, for contrast

In [6]:
b = base.answer(tc.question, tc.table_csv, False, 0.0)
print("baseline   :", repr(b["final"]), "| correct:", is_correct(b["final"], tc.gold, tc.question))
print("agent      :", repr(r["final"]), "| correct:", is_correct(r["final"], tc.gold, tc.question))
print()
print("One question proves nothing. That is what 05 is for.")

baseline   : '6.66%' | correct: True
agent      : '6.66%' | correct: True

One question proves nothing. That is what 05 is for.


## Step 7 — abstention

In [7]:
ra = A.run_agent("What is the capital of France?", tc.table_csv, toolset="code")
print("answer:", repr(ra["final"]))

answer: 'INSUFFICIENT'


## What to carry out of this notebook

```
 · an agent = a model that can call a tool, read the result, and continue
 · test tools WITHOUT the model first; a broken tool poisons every result
 · the calculator refuses anything that is not arithmetic
 · run_python exists because copying numbers out of a prompt is error-prone
 · tool_calls = 0 is a real decision, not a failure
```

Next: `05_the_experiment.ipynb` — turning single answers into a defensible claim.